In [ ]:
include("./trajopt/utils.jl")
include("./trajopt/dynamics.jl")
include("./funlopt/funl_dynamics.jl")
include("./funlopt/funl_utils.jl")
include("./funlopt/funl_constraint.jl")
include("./trajopt/scaling.jl")

In [ ]:
# load nominal trajectory
using JLD2, FileIO
@load "./data/nominal_traj_lunar_landing_N5" my_dict
xnom = my_dict["x"]
unom = my_dict["u"]
tnom = my_dict["t"];
N = size(xnom,2) - 1
dtnom = zeros(N)
for i in 1:N
    dtnom[i] = tnom[i+1]-tnom[i]
end
@assert size(xnom,2) - 1 == N

In [ ]:
Fxy_max = 1.5
Fxy_min = -1.5
Fz_max = 5.0
Fz_min = 0.0
tau_max = 0.01
tau_min = -0.01

rxmax,rymax,rzmax = 8.0,8.0,8.0
rxmin,rymin,rzmin = -8.0,-8.0,-8.0
vxmax,vymax,vzmax = 2.0,2.0,2.0
vxmin,vymin,vzmin = -2.0,-2.0,-2.0
phimax,thetamax,psimax = deg2rad(30),deg2rad(30),deg2rad(30)
phimin,thetamin,psimin = -deg2rad(30),-deg2rad(30),-deg2rad(30)
pmax,qmax,rmax = deg2rad(45),deg2rad(45),deg2rad(45)
pmin,qmin,rmin = -deg2rad(45),-deg2rad(45),-deg2rad(45)

# Glide slope angle constraints.
margin = 0.02
gamma = deg2rad(20)

list_const = [
    InputConstraint([1;0;0;0;0;0],Fxy_max),
    InputConstraint([-1;0;0;0;0;0],-Fxy_min),
    InputConstraint([0;1;0;0;0;0],Fxy_max),
    InputConstraint([0;-1;0;0;0;0],-Fxy_min),
    InputConstraint([0;0;1;0;0;0],Fz_max),
    InputConstraint([0;0;-1;0;0;0],-Fz_min),

    InputConstraint([0;0;0;1;0;0],tau_max),
    InputConstraint([0;0;0;-1;0;0],-tau_min),
    InputConstraint([0;0;0;0;1;0],tau_max),
    InputConstraint([0;0;0;0;-1;0],-tau_min),
    InputConstraint([0;0;0;0;0;1],tau_max),
    InputConstraint([0;0;0;0;0;-1],-tau_min),

    # GlideSlope(margin,gamma),

    StateConstraint([1;0;0;0;0;0;0;0;0;0;0;0],rxmax),
    # StateConstraint([-1;0;0;0;0;0;0;0;0;0;0;0],-rxmin),
    StateConstraint([0;1;0;0;0;0;0;0;0;0;0;0],rymax),
    # StateConstraint([0;-1;0;0;0;0;0;0;0;0;0;0],-rymin),
    StateConstraint([0;0;1;0;0;0;0;0;0;0;0;0],rzmax),
    # StateConstraint([0;0;-1;0;0;0;0;0;0;0;0;0],-rzmin),

    StateConstraint([0;0;0;1;0;0;0;0;0;0;0;0],vxmax),
    StateConstraint([0;0;0;-1;0;0;0;0;0;0;0;0],-vxmin),
    StateConstraint([0;0;0;0;1;0;0;0;0;0;0;0],vymax),
    StateConstraint([0;0;0;0;-1;0;0;0;0;0;0;0],-vymin),
    StateConstraint([0;0;0;0;0;1;0;0;0;0;0;0],vzmax),
    StateConstraint([0;0;0;0;0;-1;0;0;0;0;0;0],-vzmin),

    StateConstraint([0;0;0;0;0;0;1;0;0;0;0;0],phimax),
    StateConstraint([0;0;0;0;0;0;-1;0;0;0;0;0],-phimin),
    StateConstraint([0;0;0;0;0;0;0;1;0;0;0;0],thetamax),
    StateConstraint([0;0;0;0;0;0;0;-1;0;0;0;0],-thetamin),
    StateConstraint([0;0;0;0;0;0;0;0;1;0;0;0],psimax),
    StateConstraint([0;0;0;0;0;0;0;0;-1;0;0;0],-psimin),

    StateConstraint([0;0;0;0;0;0;0;0;0;1;0;0],pmax),
    StateConstraint([0;0;0;0;0;0;0;0;0;-1;0;0],-pmin),
    StateConstraint([0;0;0;0;0;0;0;0;0;0;1;0],qmax),
    StateConstraint([0;0;0;0;0;0;0;0;0;0;-1;0],-qmin),
    StateConstraint([0;0;0;0;0;0;0;0;0;0;0;1],rmax),
    StateConstraint([0;0;0;0;0;0;0;0;0;0;0;-1],-rmin),
    ]

In [ ]:
using Plots
# plotlyjs()  # Or pyplot() if you prefer
# pyplot()

# Plot setup
plt = plot(size=(400,400), legend=:topright,
           xlabel="East", ylabel="North", zlabel="Up",
           title="PDG Trajectory")

# Trajectory points and path
scatter3d!(xnom[1, :], xnom[2, :], xnom[3, :], marker=:circle, color=:orange, label="")
plot3d!(xnom[1, :], xnom[2, :], xnom[3, :], line=(:dash, :orange), label="trajectory")

# View
# Plots.camera!(plt, (azimuth=-60, elevation=20))
display(plt)
# savefig("./data_image/nominal_trajectory_lunar_landing_3D.pdf")


In [ ]:
p2 = Plots.plot(; size=(500,500))
# plot!(xprop[1,:],xprop[2,:],aspect_ratio=:equal,c=:green,linestyle=:dash,linewidth=4.0,label=nothing)
scatter!(xnom[1,:],xnom[3,:],aspect_ratio=:equal,c=:deepskyblue3,linestyle=:dash,linewidth=1.5,label=nothing)
xlabel!("X (m)")
ylabel!("Z (m)")
display(p2)

In [ ]:
dynamics = Rocket()
ix = size(xnom,1)
iu = size(unom,1)
dynamics = Rocket()

In [ ]:
A = zeros(ix,ix,N+1)
B = zeros(ix,iu,N+1)
for idx in 1:N+1
    A[:,:,idx],B[:,:,idx] = diff(dynamics,xnom[:,idx],unom[:,idx])
end

In [ ]:
# Qmax = zeros(ix,ix,N+1)
# for idx in 1:N+1
#     Qmax[:,:,idx] .= diagm([2^2,2^2,deg2rad(20)^2])
# end
# Rmax = get_Rmax_unicycle(unom,N,vmax,vmin,wmax,wmin)
# ;

In [ ]:
# include("./funlopt/funl_utils.jl")

In [ ]:
# gamma_est = Lipschitz_estimation_around_traj(N,100,xnom,unom,dynamics,Qmax,Rmax)
# gamma = gamma_est[1:N];

In [ ]:
# gamma = zeros(dynamics.iphi,N)

# Estimation of $\beta$

In [ ]:
using Interpolations

In [ ]:
u_fit = [LinearInterpolation(tnom, unom[idx,:],extrapolation_bc=Flat()) for idx in 1:iu]
A_fit = [[LinearInterpolation(tnom, A[i,j,:], extrapolation_bc=Flat()) for j in 1:ix] for i in 1:ix ]
B_fit = [[LinearInterpolation(tnom, B[i,j,:], extrapolation_bc=Flat()) for j in 1:iu] for i in 1:ix ];


In [ ]:
function model_wrapper!(f,x,p,t)
    um = p[1]
    up = p[2]
    dt = p[3]
    alpha = (dt - t) / dt
    beta = t / dt
    u1 = alpha*um + beta*up
    f .= forward(dynamics,x,u1)
end

In [ ]:
N_sampling = 20
beta = []
beta_vec = []
for i = 1:N
    tspan = (0,tnom[i+1]-tnom[i])
    saveat = range(0, stop=tnom[i+1]-tnom[i], length=N_sampling)
    prob = ODEProblem(model_wrapper!,xnom[:,i],tspan,(unom[:,i],unom[:,i+1],tnom[i+1]-tnom[i]),saveat = saveat)
    sol = solve(prob, Tsit5(), reltol=1e-9, abstol=1e-9;verbose=false);
    @assert(isapprox(sol.u[end],xnom[:,i+1];atol=0.0001))
    delta = []
    delta_vec = zeros(N_sampling,dynamics.idelta)
    for idx_sample = 1:N_sampling
        t_eval = sol.t[idx_sample] + tnom[i]
        x_eval = sol.u[idx_sample]
        u_eval = get_u_interp(t_eval,u_fit)
        A_eval,B_eval = diff(dynamics,x_eval,u_eval)
        eA = get_ABF_interp(t_eval,A_fit,ix,ix) .- A_eval
        eB = get_ABF_interp(t_eval,B_fit,ix,iu) .- B_eval

        delta_ = [eA eB]
        push!(delta,opnorm(delta_,2))
        delta_vec[idx_sample,:] .= [
                                    norm([eA[4,7] eA[4,8] eA[4,9] eB[4,1] eB[4,2] eB[4,3]])
                                    norm([eA[5,7] eA[5,8] eA[5,9] eB[5,1] eB[5,2] eB[5,3]])
                                    norm([eA[6,7] eA[6,8] eB[6,1] eB[6,2] eB[6,3]])
                                    norm([eA[7,7] eA[7,8] eA[7,11] eA[7,12]])
                                    norm([eA[8,7] eA[8,11] eA[8,12]])
                                    norm([eA[9,7] eA[9,8] eA[9,11] eA[9,12]])
                                    norm([eA[10,11] eA[10,12]])
                                    norm([eA[11,10] eA[11,12]])
                                    norm([eA[12,10] eA[12,11]])
                                    ]
    end
    # push!(beta,maximum(delta))
    push!(beta,diagm(maximum(delta_vec,dims=1)[1,:]))
end

In [ ]:
beta = beta
# Temporary.
gamma = []
for i in 1:N
    push!(gamma,0.0 * beta[i])
end

In [ ]:
dynamics.C = [dynamics.Co;dynamics.Co]
dynamics.D = [dynamics.Do;dynamics.Do]
dynamics.E = [dynamics.Eo dynamics.Eo]

In [ ]:
xmin = [0,0,0, 0,0,0, 0,0,0, 0,0,0];
xmax = [1,1,1, 0.5,0.5,0.5, deg2rad(1),deg2rad(1),deg2rad(1), deg2rad(1),deg2rad(1),deg2rad(1)];
umin = [0,0,0, 0,0,0];
umax = [1,1,1, 0.01,0.01,0.01];

scaler = Scaling(xmin, xmax, umin, umax, 1e4 * Matrix{Float64}(1.0I,dynamics.ilam,dynamics.ilam),tnom[end],0,0)

In [ ]:
include("./funlopt/funl_synthesis.jl")

In [ ]:
# lambda_w = 2.0
# list_lambda_w = [0.01,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,1.0]#,1.2,1.4,1.6,1.8,2.0]
list_lambda_w = [0.0]
list_solve_time = []
cost_lambda_w = []
list_fs = []
println("Line search for lambda_w from 0.01 to 2.0")
for lambda_w in list_lambda_w
        fs = FunnelSynthesis(N,lambda_w,dynamics,list_const,scaler,verbosity=true,flag_copositivity_type=0)
        cost,solve_time = run!(fs,
        gamma,
        beta,
        xnom,unom,tnom,"Mosek")
        println("lambda_w: ",lambda_w, " cost: ",abs(cost) < 1e-6 ? "diverged" : cost)
        push!(cost_lambda_w,cost)
        push!(list_fs,fs)
        push!(list_solve_time,solve_time)
end
idx_lambda_w = 1:length(list_lambda_w)
idx_converged = abs.(cost_lambda_w) .>= 1e-6
cost_lambda_w_converged = cost_lambda_w[idx_converged]
idx_fs = argmin(cost_lambda_w_converged)
idx_min = idx_lambda_w[idx_converged][idx_fs]

lambda_w = list_lambda_w[idx_min]
cost = cost_lambda_w[idx_min]
fs = list_fs[idx_min]
solve_time = list_solve_time[idx_min]

println(lambda_w," is picked. The corresponding cost is ",cost,". The solve time is ",solve_time,".")

In [ ]:
Qnom,Ynom,lamnom = fs.solution.Q,fs.solution.Y,fs.solution.lam

In [ ]:
p2 = Plots.plot(; size=(500,500),guidefontsize=15,legendfontsize=10)
# plot!(xprop[1,:],xprop[3,:],aspect_ratio=:equal,c=:green,linestyle=:dash,linewidth=1.5,label="nominal")
scatter!(xnom[1,:],xnom[3,:],aspect_ratio=:equal,c=:deepskyblue3,linestyle=:dash,linewidth=1.5,label=nothing)
for idx in 1:N+1
    label = nothing
    if idx == 1
        label = "funnel"
    end
    plot_ellipse(p2,Qnom[:,:,idx],xnom[:,idx],"deepskyblue3",label=label,idx1=1,idx2=3)
end
xlabel!("Easy")
ylabel!("Up")
display(p2)

In [ ]:
fs2 = FunnelSynthesis(N,lambda_w,dynamics,list_const,scaler,verbosity=true,flag_copositivity_type=0)
cost,solve_time = run!(fs2,
gamma,
beta,
xnom,unom,tnom,"Mosek")
println("lambda_w: ",lambda_w, " cost: ",cost," solve time ",solve_time)

In [ ]:
Qnom_,Ynom_,lamnom_ = fs.solution.Q,fs.solution.Y,fs.solution.lam
Qnom,Ynom,lamnom = fs2.solution.Q,fs2.solution.Y,fs2.solution.lam

In [ ]:
p2 = Plots.plot(; size=(500,500),guidefontsize=15,legendfontsize=10)
# plot!(xprop[1,:],xprop[3,:],aspect_ratio=:equal,c=:green,linestyle=:dash,linewidth=1.5,label="nominal")
scatter!(xnom[1,:],xnom[3,:],aspect_ratio=:equal,c=:deepskyblue3,linestyle=:dash,linewidth=1.5,label=nothing)
for idx in 1:N+1
    label = nothing
    if idx == 1
        label = "funnel"
    end
    plot_ellipse(p2,Qnom[:,:,idx],xnom[:,idx],"deepskyblue3",label=label,idx1=1,idx2=3)
end
xlabel!("Easy")
ylabel!("Up")
display(p2)

In [ ]:
p = []
ylabel_list = ["East","North","Up","Vx","Vy","Vz","roll","pitch","yaw","p","q","r"]
for idx in 1:12
    push!(p,plot(tnom,xnom[idx,:],seriestype = :scatter,label=""))
    # plot!(tprop,xprop[idx,:],label="")
    plot!(tnom,xnom[idx,:] + sqrt.(Qnom[idx,idx,:]),color=:blue,seriestype = :scatter,label="")
    # plot!(tprop,xprop[idx,:] + sqrt.(Qprop[idx,idx,:]),color=:blue,label="")
    plot!(tnom,xnom[idx,:] - sqrt.(Qnom[idx,idx,:]),color=:blue,seriestype = :scatter,label="")
    # plot!(tprop,xprop[idx,:] - sqrt.(Qprop[idx,idx,:]),color=:blue,label="")
    if idx == 12
        xlabel!("Time (s)")
    end
    # ylabel!("x" * string(idx))
    ylabel!(ylabel_list[idx])
end
plot(p[1:12]..., layout = (6, 2), size = (1000, 1200))

In [ ]:
# Set default plot settings for academic paper
default(fontfamily="Times",  # Use Times New Roman font
        titlefont=14,        # Title font size
        guidefont=12,        # Label (guide) font size
        tickfont=10,         # Tick font size
        legendfont=12,       # Legend font size
        grid=true)          # Remove grid lines for a cleaner look


In [ ]:
input_proj_funl_node,Rnom = project_onto_input(Qnom,Ynom)

In [ ]:
p = []
for idx in 1:6
    push!(p,plot(tnom,unom[idx,:],linestyle=:dash,color="black",label=""))
    scatter!(tnom,unom[idx,:]+input_proj_funl_node[idx],color="deepskyblue3",label="")
    scatter!(tnom,unom[idx,:]-input_proj_funl_node[idx],color="deepskyblue3",label="")
    # plot!(tprop,uprop[idx,:]+input_proj_funl_prop[idx],color="deepskyblue3",label="")
    # plot!(tprop,uprop[idx,:]-input_proj_funl_prop[idx],color="deepskyblue3",label="")
    # plot!(tnom,tnom*0 .+ vmax,linestyle=:dash,color="red",label="")
    # plot!(tnom,tnom*0 .+ vmin,linestyle=:dash,color="red",label="")
end
plot(p[1:6]...,layout=(3,2),size=(800,500))

# samples

In [ ]:
Q_fit = [[LinearInterpolation(tnom, Qnom[i,j,:], extrapolation_bc=Flat()) for j in 1:ix] for i in 1:ix ]
Y_fit = [[LinearInterpolation(tnom, Ynom[i,j,:], extrapolation_bc=Flat()) for j in 1:ix] for i in 1:iu ]
;

In [ ]:
include("./funlopt/funl_utils.jl")

In [ ]:
num_sample = 500
xs_list = []
for i in 1:num_sample
    z = randn(ix)
    z = z / norm(z)
    push!(xs_list,xnom[:,1] + sqrt(Qnom[:,:,1]) * z)
end

In [ ]:
xsam_fwd,tsam,xsam,usam,xnomprop = [],[],[],[],[]
i = 0
for xs in xs_list
    # println(i)
    i += 1
    xf_,ts_,xsam_,usam_,xnom_ = propagate_from_funnel_entry(xs,
        dynamics,
        xnom,unom,tnom,
        Qnom,Ynom)
    push!(xsam_fwd,xf_)
    push!(tsam,ts_)
    push!(xsam,xsam_)
    push!(usam,usam_)
    push!(xnomprop,xnom_)
end

In [ ]:
p2 = Plots.plot(; size=(500,500),guidefontsize=15,legendfontsize=10)
# plot!(xprop[1,:],xprop[3,:],aspect_ratio=:equal,c=:green,linestyle=:dash,linewidth=1.5,label=nothing)
scatter!(xnom[1,:],xnom[3,:],aspect_ratio=:equal,c=:deepskyblue3,linestyle=:dash,linewidth=1.5,label=nothing)
for idx in 1:N+1
    label = nothing
    if idx == 1
        label = "funnel"
    end
    plot_ellipse(p2,Qnom[:,:,idx],xnom[:,idx],"deepskyblue3",label=label,idx1=1,idx2=3)
end
for x_ in xsam
    plot!(x_[1,:],x_[3,:],color="purple",label=nothing)
end
xlabel!("East")
ylabel!("Up")
display(p2)

In [ ]:
# Lyapunov function
Lypsam = []
idx = 1
for idx = 1:length(tsam)
    tsam_ = tsam[idx]
    xsam_ = xsam[idx]
    xprop_ = xnomprop[idx]
    Lyp = []
    for iode in 1:length(tsam_)
        t_ = tsam_[iode]
        x_ = xsam_[:,iode]
        xnom_ = xprop_[:,iode]
        Q_ = get_ABF_interp(t_,Q_fit,ix,ix)
        val = (x_ - xnom_)' * inv(Q_) * (x_ - xnom_)
        push!(Lyp,val)
    end
    push!(Lypsam,Lyp)
end


In [ ]:
p3 = Plots.plot(; size=(500,300))
for idx = 1:length(tsam)
    plot!(tsam[idx],Lypsam[idx],c=:purple,label="")
end
plot!(tsam[1],tsam[1]*0 .+ 1.0,c=:red,linestyle=:dash,label="")
ylims!(p3, 0.0, 1.5)
display(p3)

In [ ]:
funl_dict = Dict("Q1" => Qnom_, "Y1" => Ynom_, "lam1" => lamnom_, "Q2" => Qnom, "Y2" => Ynom, "lam" => lamnom)

using JLD2, FileIO
@save "./data/funnel_result_lunar_landing_N" * string(N) funl_dict